# 第22章　整形外科 ― X線骨折診断

**『医療診断支援AI開発　実装編 ― 本格実装（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 22.6　ハイブリッドな2段階パイプライン

In [ ]:
# 2段階パイプラインの推論フロー（擬似コード）
def diagnose_fracture(xray_image):
    # 第1段階：スクリーニング（EfficientNet-B3）
    # 評価するときは、ここで止まった症例も必ず分母に残す。第2段へ進んだ症例だけで
    # 精度を測ると、第1段が落とした骨折が集計から消え、成績が実力より良く見える。
    prob_fracture = classifier(xray_image).sigmoid()
    if prob_fracture < THRESHOLD:      # 閾値は感度優先で低めに設定
        return {"fracture": False, "prob": prob_fracture, "stage_stopped": 1}

    # 第2段階：位置特定（YOLO11）
    boxes = detector(xray_image)       # 骨折位置のバウンディングボックス

    # 第3段階：可視化（Grad-CAM）
    heatmap = grad_cam(classifier, xray_image)

    return {"fracture": True, "prob": prob_fracture,
            "boxes": boxes, "heatmap": heatmap}

## 追加ケース ― 骨盤骨折と、椎体圧迫骨折の定量

In [ ]:
def genant_grade(ant_h, mid_h, post_h, ref_h):
    """高さ減少に基づく簡易区分。正式なGenant分類や骨折診断の代用ではない。
    ref_h: 参照高。上下の健常椎体の後縁高（または同一患者の正常椎体の平均）。
    椎体内の比だけでは全体が一様に低くなる変形を捉えられないため、
    参照高との比も用いる。grade=0やshape="normal"は、この数値規則での区分を示す。"""
    import math, numbers
    heights = (ant_h, mid_h, post_h, ref_h)
    if not all(isinstance(v, numbers.Real) and not isinstance(v, bool)
               and math.isfinite(v) and v > 0 for v in heights):
        return {"grade": None, "height_loss": None, "shape": None,
                "state": "評価不能（高さが数値でない、非有限、または0以下）"}
    ant_h, mid_h, post_h, ref_h = (float(v) for v in heights)
    def snap_boundary(value):
        # ごく小さな計算誤差だけを境界へそろえる。計測誤差の許容幅ではない。
        for boundary in (0.20, 0.25, 0.40):
            if math.isclose(value, boundary, rel_tol=0.0, abs_tol=1e-12):
                return boundary
        return value
    hmax = max(ant_h, mid_h, post_h)             # その椎体の基準高
    loss_within = 1 - min(ant_h, mid_h) / hmax   # 椎体内の変形（楔状・中央陥凹）
    loss_ref = snap_boundary(1 - min(ant_h, mid_h, post_h) / ref_h)
    loss = snap_boundary(max(loss_within, loss_ref))
    grade = 0 if loss < 0.20 else 1 if loss < 0.25 else 2 if loss <= 0.40 else 3
    # 形の判定は「中央がへこんでいるか」を先に見る。
    # ant_h < post_h を最初に置くと、前縁がわずかでも低いだけで常にwedgeになり、
    # biconcave（中央陥凹型）が永久に検出されなくなる。
    if mid_h < 0.95 * min(ant_h, post_h):          # 中央が明らかに低い
        shape = "biconcave"
    elif ant_h < 0.95 * post_h:                    # 前縁が明らかに低い
        shape = "wedge"
    elif loss_ref >= 0.20:                         # 椎体内の形は保ったまま、全体が低い
        shape = "crush"
    else:
        shape = "normal"                           # 参照高と比べても減っていない
    return {"grade": grade, "height_loss": round(loss, 2), "shape": shape}
# Grade1: 20-25%, Grade2: 25-40%, Grade3: >40% の高さ減少

## 骨折の先へ ― 変性した脊椎を定量する

In [ ]:
import math, numbers

def meyerding_grade(slip_mm, lower_body_width_mm):
    # 計測値が数値として有効か（NaN でないか）を先に見る。NaN は比較がすべて偽になり、
    # 検査しないと NaN が Grade 5・state=ok という有効な区分に化ける
    # numbers.Real なら NumPy の整数・浮動小数スカラー（mask.sum() の返り値など）も通る。bool は数値扱いしない
    if not all(isinstance(v, numbers.Real) and not isinstance(v, bool) and math.isfinite(v)
               for v in (slip_mm, lower_body_width_mm)):
        return {"slip_pct": None, "meyerding": None, "state": "評価不能（計測値が数値でない、または非有限）"}
    # 分母が取れなければ「評価不能」。1e-6 で置き換えると、ずれ率が数億%になって返る。
    if lower_body_width_mm <= 0:
        return {"slip_pct": None, "meyerding": None, "state": "評価不能（椎体幅が測れない）"}
    if slip_mm <= 0:                                          # すべりが無ければグレードは付かない
        return {"slip_pct": 0.0, "meyerding": 0, "state": "すべりなし"}
    pct = 100.0 * slip_mm / lower_body_width_mm               # 下位椎体幅に対するずれ率
    # 区切りは 25 / 50 / 75 / 100%。境界値をどちらへ入れるかは採用する定義に合わせ、
    # ここでは「25%ちょうどはGrade 1」（各区間の上端を含める）に統一する。
    grade = 1 if pct <= 25 else 2 if pct <= 50 else 3 if pct <= 75 else 4 if pct <= 100 else 5
    return {"slip_pct": round(pct, 1), "meyerding": grade, "state": "ok"}   # 5 = spondyloptosis